<a href="https://colab.research.google.com/github/mbmanoj904-hue/Generative-ai/blob/main/langgraph_2_3Llm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
# ============================================================
# LANGGRAPH + RAG + 3 LLM FAULT-TOLERANT SYSTEM
# GOOGLE COLAB - RUN EVERYTHING IN ONE CELL
# ============================================================

# ============================================================
# 1. INSTALL REQUIRED LIBRARIES
# ============================================================

!pip install -q -U \
    langgraph \
    langchain \
    langchain-community \
    langchain-text-splitters \
    langchain-huggingface \
    faiss-cpu \
    sentence-transformers \
    google-genai


# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import os
from typing import TypedDict

from google import genai
from google.colab import userdata

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph, START, END


# ============================================================
# 3. GEMINI API KEY
# ============================================================

# Supports both possible Colab secret names
api_key = (
   userdata.get("Geminiai")
)

if not api_key:
    raise ValueError(
        "Gemini API key not found.\n"
        "Add your key in Colab Secrets as GEMINI_API_KEY "
        "or Geminiai."
    )

client = genai.Client(api_key=api_key)

# IMPORTANT:
# Old model: gemini-2.5-flash  ❌
# New model: gemini-3.8-flash  ✅
MODEL_NAME = "gemini-3.6-flash"

print("Gemini client initialized")
print("Model:", MODEL_NAME)


# ============================================================
# 4. KNOWLEDGE BASE
# ============================================================

knowledge = [
    """
    Artificial Intelligence (AI) is the field of computer science
    that focuses on creating machines and software capable of
    performing tasks that normally require human intelligence.

    Examples include learning, reasoning, problem solving,
    understanding language, computer vision and decision making.
    """,

    """
    Machine Learning (ML) is a subset of Artificial Intelligence.
    Machine Learning allows computers to learn patterns from data
    and make predictions or decisions without being explicitly
    programmed for every individual task.
    """,

    """
    Deep Learning is a subset of Machine Learning that uses
    artificial neural networks with multiple layers.
    Deep learning is widely used for image recognition,
    speech recognition, natural language processing and
    generative AI.
    """,

    """
    Large Language Models (LLMs) are AI models trained on large
    amounts of text data. They can understand and generate
    human-like text. Examples of tasks include question answering,
    summarization, translation and code generation.
    """,

    """
    Retrieval-Augmented Generation (RAG) combines information
    retrieval with a language model.

    First, relevant information is retrieved from a knowledge
    base. Then the retrieved information is provided to the
    language model so that it can generate a more grounded answer.
    """,

    """
    LangGraph is a framework for building stateful applications
    and agent workflows using graphs.

    A LangGraph workflow contains nodes and edges. Nodes perform
    tasks, while edges define the flow between tasks.
    """,

    """
    FAISS is a library for efficient similarity search of vectors.
    It is commonly used in RAG systems to store embeddings and
    retrieve documents that are semantically similar to a query.
    """
]

documents = [
    Document(page_content=text)
    for text in knowledge
]

print("\nDocuments:", len(documents))


# ============================================================
# 5. SPLIT DOCUMENTS INTO CHUNKS
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))


# ============================================================
# 6. CREATE HUGGING FACE EMBEDDINGS
# ============================================================

print("\nLoading embedding model...")

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded")


# ============================================================
# 7. CREATE FAISS VECTOR DATABASE
# ============================================================

print("\nCreating FAISS vector database...")

vector_db = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS vector database created")


# ============================================================
# 8. LANGGRAPH STATE
# ============================================================

class GraphState(TypedDict):
    question: str
    context: str

    llm1_answer: str
    llm2_answer: str
    llm3_answer: str

    llm1_status: str
    llm2_status: str
    llm3_status: str

    final_answer: str


# ============================================================
# 9. RAG RETRIEVAL NODE
# ============================================================

def retrieve_node(state: GraphState):

    question = state["question"]

    print("\n" + "=" * 60)
    print("RETRIEVING RELEVANT INFORMATION")
    print("=" * 60)

    try:

        results = vector_db.similarity_search(
            question,
            k=3
        )

        context = "\n\n".join(
            doc.page_content
            for doc in results
        )

        print("Retrieved documents:", len(results))

        return {
            "context": context
        }

    except Exception as e:

        print("Retrieval failed:", str(e))

        return {
            "context": ""
        }


# ============================================================
# 10. GEMINI LLM FUNCTION
# ============================================================

def call_llm(prompt):

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    if not response.text:
        raise ValueError("Empty response received from Gemini")

    return response.text.strip()


# ============================================================
# 11. LLM 1 - ANSWER GENERATOR
# ============================================================

def llm1_node(state: GraphState):

    print("\n" + "=" * 60)
    print("LLM 1 - ANSWER GENERATOR")
    print("=" * 60)

    question = state["question"]
    context = state["context"]

    prompt = f"""
You are LLM 1, the primary answer generator.

Answer the user's question using the provided context.

Question:
{question}

Context:
{context}

Instructions:
- Give a clear and simple answer.
- Use the context whenever possible.
- Do not invent facts.
- Keep the answer concise.
"""

    try:

        answer = call_llm(prompt)

        print("LLM 1 SUCCESS")

        return {
            "llm1_answer": answer,
            "llm1_status": "SUCCESS"
        }

    except Exception as e:

        print("LLM 1 FAILED")
        print("Error:", str(e))

        return {
            "llm1_answer": "",
            "llm1_status": "FAILED"
        }


# ============================================================
# 12. LLM 2 - CHECKER / CORRECTOR
# ============================================================

def llm2_node(state: GraphState):

    print("\n" + "=" * 60)
    print("LLM 2 - CHECKER / CORRECTOR")
    print("=" * 60)

    question = state["question"]
    context = state["context"]
    llm1_answer = state["llm1_answer"]

    # If LLM1 failed, LLM2 independently generates an answer
    if llm1_answer:

        previous_answer = llm1_answer

        task = f"""
Review the answer generated by LLM 1.

LLM 1 Answer:
{previous_answer}

Check it against the context.
Correct any incorrect or missing information.
Return the improved answer only.
"""

    else:

        task = """
LLM 1 failed.

Generate the answer yourself using the context.
Return a clear and accurate answer.
"""

    prompt = f"""
You are LLM 2, the checker and corrector.

Question:
{question}

Context:
{context}

{task}

Do not mention internal processing.
"""

    try:

        answer = call_llm(prompt)

        print("LLM 2 SUCCESS")

        return {
            "llm2_answer": answer,
            "llm2_status": "SUCCESS"
        }

    except Exception as e:

        print("LLM 2 FAILED")
        print("Error:", str(e))

        return {
            "llm2_answer": "",
            "llm2_status": "FAILED"
        }


# ============================================================
# 13. LLM 3 - FINAL ANSWER / BACKUP
# ============================================================

def llm3_node(state: GraphState):

    print("\n" + "=" * 60)
    print("LLM 3 - FINAL ANSWER / BACKUP")
    print("=" * 60)

    question = state["question"]
    context = state["context"]

    llm1_answer = state["llm1_answer"]
    llm2_answer = state["llm2_answer"]

    # Prefer LLM2 if available
    if llm2_answer:

        best_answer = llm2_answer

    elif llm1_answer:

        best_answer = llm1_answer

    else:

        best_answer = "No previous LLM answer was available."

    prompt = f"""
You are LLM 3, the final answer generator and backup LLM.

Your job is to ALWAYS try to produce the final answer.

Question:
{question}

Retrieved Context:
{context}

Previous LLM 1 Answer:
{llm1_answer if llm1_answer else "LLM 1 failed"}

Previous LLM 2 Answer:
{llm2_answer if llm2_answer else "LLM 2 failed"}

Preferred previous answer:
{best_answer}

Instructions:
- Produce the final answer for the user.
- Correct mistakes if necessary.
- Use the retrieved context.
- If previous answers are unavailable, answer using the context.
- Keep the answer clear and simple.
- Do not mention internal LLM failures.
"""

    try:

        answer = call_llm(prompt)

        print("LLM 3 SUCCESS")

        return {
            "llm3_answer": answer,
            "llm3_status": "SUCCESS",
            "final_answer": answer
        }

    except Exception as e:

        print("LLM 3 FAILED")
        print("Error:", str(e))

        # Final local fallback
        if llm2_answer:

            fallback = llm2_answer

        elif llm1_answer:

            fallback = llm1_answer

        elif context:

            fallback = (
                "Based on the retrieved information:\n\n"
                + context
            )

        else:

            fallback = "Unable to generate an answer."

        return {
            "llm3_answer": fallback,
            "llm3_status": "FAILED",
            "final_answer": fallback
        }


# ============================================================
# 14. BUILD LANGGRAPH
# ============================================================

workflow = StateGraph(GraphState)

workflow.add_node(
    "retrieve",
    retrieve_node
)

workflow.add_node(
    "llm1",
    llm1_node
)

workflow.add_node(
    "llm2",
    llm2_node
)

workflow.add_node(
    "llm3",
    llm3_node
)


# ============================================================
# 15. GRAPH FLOW
# ============================================================

workflow.add_edge(
    START,
    "retrieve"
)

workflow.add_edge(
    "retrieve",
    "llm1"
)

workflow.add_edge(
    "llm1",
    "llm2"
)

workflow.add_edge(
    "llm2",
    "llm3"
)

workflow.add_edge(
    "llm3",
    END
)


# ============================================================
# 16. COMPILE GRAPH
# ============================================================

app = workflow.compile()

print("\nLangGraph compiled successfully")


# ============================================================
# 17. USER QUESTION
# ============================================================

question = input(
    "\nEnter your question: "
)


# ============================================================
# 18. RUN LANGGRAPH
# ============================================================

initial_state = {
    "question": question,
    "context": "",

    "llm1_answer": "",
    "llm2_answer": "",
    "llm3_answer": "",

    "llm1_status": "",
    "llm2_status": "",
    "llm3_status": "",

    "final_answer": ""
}

result = app.invoke(initial_state)


# ============================================================
# 19. DISPLAY RESULTS
# ============================================================

print("\n")
print("=" * 70)
print("LLM STATUS")
print("=" * 70)

print("LLM 1:", result["llm1_status"])
print("LLM 2:", result["llm2_status"])
print("LLM 3:", result["llm3_status"])


print("\n")
print("=" * 70)
print("FINAL ANSWER")
print("=" * 70)

print(result["final_answer"])

print("\n")
print("=" * 70)
print("SYSTEM COMPLETED")
print("=" * 70)

Gemini client initialized
Model: gemini-3.6-flash

Documents: 7
Chunks: 7

Loading embedding model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding model loaded

Creating FAISS vector database...
FAISS vector database created

LangGraph compiled successfully

Enter your question: what is dl

RETRIEVING RELEVANT INFORMATION
Retrieved documents: 3

LLM 1 - ANSWER GENERATOR
LLM 1 SUCCESS

LLM 2 - CHECKER / CORRECTOR
LLM 2 SUCCESS

LLM 3 - FINAL ANSWER / BACKUP
LLM 3 SUCCESS


LLM STATUS
LLM 1: SUCCESS
LLM 2: SUCCESS
LLM 3: SUCCESS


FINAL ANSWER
Based on the provided context, **DL** stands for **Deep Learning**. 

Deep Learning is a subset of Machine Learning that uses artificial neural networks with multiple layers. It is widely used for:
- Image recognition
- Speech recognition
- Natural language processing
- Generative AI


SYSTEM COMPLETED
